In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from utils.db_interface import get_engine
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from baseline_model.utils.calibrated_logreg_model import CalibratedLogRegModel

### Config

In [ ]:
UNCALIBRATED_MODELS_DIR = "../models/uncalibrated"
CALIBRATED_MODELS_DIR = "../models/calibrated"

### Setup

In [ ]:
os.makedirs(UNCALIBRATED_MODELS_DIR, exist_ok=True)
os.makedirs(CALIBRATED_MODELS_DIR, exist_ok=True)

### Help Functions

In [ ]:
def load_and_prepare_data_baseline():
    """
    Load the last MAP value (last_map) and binary event label from the database.
    Splits: train / val / test
    """
    engine = get_engine()
    
    query = """
    SELECT
        mw.subject_id,
        (elem ->> 'value')::float AS last_map,
        mw.positive_event,
        sm.split
    FROM ce_approach.mix_windows mw
    LEFT JOIN LATERAL (
        SELECT elem
        FROM unnest(mw.map_values) AS elem
        ORDER BY (elem ->> 'pos')::float DESC
        LIMIT 1
    ) last_map_elem ON TRUE
    JOIN ce_approach.split_all_subjects sm
    ON mw.subject_id = sm.subject_id
    WHERE sm.split IN ('train', 'val', 'test')
    """
    df = pd.read_sql(query, engine)
    df["label"] = df["positive_event"].astype(int)

    # Create train / val datasets
    X_train = df[df["split"] == "train"][["last_map"]]
    y_train = df[df["split"] == "train"]["label"]

    X_val = df[df["split"] == "val"][["last_map"]]
    y_val = df[df["split"] == "val"]["label"]

    return X_train, y_train, X_val, y_val

In [ ]:
def calibrate_logreg_isotonic(model, X_val, y_val):
    """
    Fit Isotonic Regression to calibrate Logistic Regression probabilities.
    """
    # Get predicted probabilities for validation set
    y_val_pred = model.predict_proba(X_val)[:, 1]

    # Sort predictions (required for isotonic regression)
    sorted_idx = np.argsort(y_val_pred)

    # Fit isotonic calibrator
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_val_pred[sorted_idx], y_val.iloc[sorted_idx])
    return iso

### Train Baseline Model

In [ ]:
# Load train/val data
X_train, y_train, X_val, y_val = load_and_prepare_data_baseline()

# 1. Train uncalibrated Logistic Regression
model = LogisticRegression()
model.fit(X_train, y_train)

# Save uncalibrated model
joblib.dump(model, os.path.join(UNCALIBRATED_MODELS_DIR, "baseline_lr.pkl"))
print("Uncalibrated baseline model saved.")

### Calibrate Baseline Model

In [ ]:
# 2. Calibrate using validation data
iso = calibrate_logreg_isotonic(model, X_val, y_val)
calibrated_model = CalibratedLogRegModel(model, iso)

# Save calibrated model
joblib.dump(calibrated_model, os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr.pkl"))
print("Calibrated baseline model saved.")